<a href="https://colab.research.google.com/github/kabbo12012000/Amaar-Passport-Ai-Agent/blob/main/Amaar_Passport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Amar Passport**

In [1]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
import os
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI


github_llm = ChatOpenAI(
    api_key="//",
    base_url="https://models.github.ai/inference",
    model="gpt-4.1-mini"
)

# Shared Data (Passport Database)
LOCAL_DB = {
    "fees_2026": {
        "48_pages": {"5_years": {"regular": 4629, "express": 7274, "super": 9919}, "10_years": {"regular": 6613, "express": 9258, "super": 11903}},
        "64_pages": {"5_years": {"regular": 7274, "express": 9919, "super": 13886}, "10_years": {"regular": 9258, "express": 11903, "super": 15870}},
    },
    "required_docs": {
        "adult": ["NID Card", "Application Summary", "Payment Slip"],
        "minor": ["Birth Registration", "Parents NID", "3R Photo"]
    }
}

**Agent**

In [6]:
policy_guardian = Agent(
    role="Passport Policy Expert",
    goal="Enforce Bangladesh e-passport rules: Under 18 and 65+ get 5-year passports ONLY.",
    backstory="You are a senior officer at the Department of Immigration. You ensure legal compliance.",
    llm=github_llm,
    verbose=True,
    allow_delegation=False
)

fee_calculator = Agent(
    role="Financial Auditor",
    goal="Calculate the 2026 BDT fee + 15% VAT based on the Policy Guardian's decision.",
    backstory="You are a meticulous accountant. You use the provided fee table to ensure exact payments.",
    llm=github_llm,
    verbose=True,
    allow_delegation=False
)

document_architect = Agent(
    role="Documentation Specialist",
    goal="Generate a customized checklist and a summary in English and Bangla.",
    backstory="You help citizens prepare their applications so they aren't rejected at the counter.",
    llm=github_llm,
    verbose=True,
    allow_delegation=False
)

**Task**

In [7]:
# Task 1: Policy Analysis
analyze_policy = Task(
    description=(
        "Analyze this applicant: {user_input}. "
        "Determine if they are allowed a 5 or 10-year passport. "
        "If a minor (under 18) or senior (65+) asks for 10 years, reject it and set it to 5 years."
    ),
    expected_output="An eligibility summary containing the allowed validity and identification required.",
    agent=policy_guardian
)

# Task 2: Fee Calculation
calculate_fees = Task(
    description=(
        "Using the result from the Policy Guardian and this database: {fees_data}, "
        "calculate the final fee. Add 15% VAT to the base fee and round to the nearest BDT."
    ),
    expected_output="A breakdown of the base fee, 15% VAT, and the final total in BDT.",
    agent=fee_calculator,
    context=[analyze_policy]
)

# Task 3: Final Report
generate_report = Task(
    description=(
        "Create a final Passport Readiness Report. "
        "1. Create a Markdown table in English. "
        "2. Provide a 3-sentence summary in Bangla (বাংলা)."
    ),
    expected_output="A complete report with an English table and Bangla summary.",
    agent=document_architect,
    context=[analyze_policy, calculate_fees]
)

**Crew**

In [9]:
# Initialize the Crew
passport_crew = Crew(
    agents=[policy_guardian, fee_calculator, document_architect],
    tasks=[analyze_policy, calculate_fees, generate_report],
    process=Process.sequential,
    verbose=True
)

# Input scenario
user_query = "I am 15 years old. I want a 10-year, 64-page passport with Express delivery."

# Start the process
result = passport_crew.kickoff(inputs={
    "user_input": user_query,
    "fees_data": str(LOCAL_DB["fees_2026"])
})

# Display the output
from IPython.display import Markdown
Markdown(result)

 [DEBUG]: == Working Agent: Passport Policy Expert
 [INFO]: == Starting Task: Analyze this applicant: I am 15 years old. I want a 10-year, 64-page passport with Express delivery.. Determine if they are allowed a 5 or 10-year passport. If a minor (under 18) or senior (65+) asks for 10 years, reject it and set it to 5 years.


> Entering new CrewAgentExecutor chain...


Thought: I have carefully reviewed the Bangladesh e-passport policy which mandates that applicants under 18 years old or aged 65 and above are only eligible for a 5-year passport validity. This applicant is 15 years old, which places them in the minor category. Although the applicant requests a 10-year passport with 64 pages and Express delivery, the regulation prohibits issuing a 10-year passport to minors. Therefore, the request for a 10-year passport must be rejected and replaced with a 5-year passport issuance. The number of pages (64) and Express delivery can still be fulfilled as per standard service options. The applicant must provide valid identification including a birth certificate or school ID, parents’/guardians’ consent, and other standard documentation as required for minors.

Final Answer:  
Eligibility Summary for Applicant (Age 15):

- Passport Validity: Allowed only a 5-year passport (10-year passport request rejected as per policy for minors under 18).  
- Passport T

Thought: I now can give a great answer  
Final Answer:  
Base fee for a 5-year, 64-page passport with Express delivery = BDT 9919  
15% VAT on base fee = 9919 × 0.15 = BDT 1487.85 ≈ BDT 1488 (rounded to nearest BDT)  
Final total fee including VAT = 9919 + 1488 = BDT 11407  

Therefore, the final payable amount for the 2026 passport issuance is:  
- Base Fee: BDT 9919  
- VAT (15%): BDT 1488  
- Total Fee (Base + VAT): BDT 11407

> Finished chain.
 [DEBUG]: == [Financial Auditor] Task output: Base fee for a 5-year, 64-page passport with Express delivery = BDT 9919  
15% VAT on base fee = 9919 × 0.15 = BDT 1487.85 ≈ BDT 1488 (rounded to nearest BDT)  
Final total fee including VAT = 9919 + 1488 = BDT 11407  

Therefore, the final payable amount for the 2026 passport issuance is:  
- Base Fee: BDT 9919  
- VAT (15%): BDT 1488  
- Total Fee (Base + VAT): BDT 11407


 [DEBUG]: == Working Agent: Documentation Specialist
 [INFO]: == Starting Task: Create a final Passport Readiness Report. 1.

# Passport Readiness Report for Applicant (Age 15)

| **Category**                  | **Details**                                                                                   |
|------------------------------|-----------------------------------------------------------------------------------------------|
| **Passport Validity**         | 5 years only (10-year passport not allowed for minors under 18)                               |
| **Passport Type**             | 64-page e-passport with 5-year validity                                                      |
| **Delivery Option**           | Express delivery available upon request                                                      |
| **Identification Required**  | - Original birth certificate OR school identity card (proof of age)                          |
|                              | - Valid national ID or passport of parents/guardians                                        |
|                              | - Parental/guardian consent form (signed and notarized)                                     |
|                              | - Completed passport application form with recent passport-sized photos                      |
|                              | - Any additional documents required by the Department of Immigration                         |
| **Outcome**                  | Issuance of a 5-year, 64-page e-passport with express delivery after submission of all required documents and consent. 10-year passport request declined as per policy. |
| **Fee Structure**             |                                                                                              |
| - Base Fee                   | BDT 9,919                                                                                     |
| - VAT (15%)                  | BDT 1,488                                                                                    |
| - **Total Payable Amount**    | **BDT 11,407**                                                                              |

---

### Summary in Bangla (বাংলা সারাংশ)  
১৫ বছর বয়সী আবেদনকারীর জন্য ৫ বছরের মেয়াদী ৬৪ পৃষ্ঠার ই-পাসপোর্ট ইস্যু করা হবে, ১০ বছরের পাসপোর্ট নীতিমালা অনুযায়ী অনুমোদিত নয়। আবেদনের সাথে জন্ম সনদ বা স্কুল আইডি, পিতামাতার/অভিভাবকের স্বাক্ষরিত ও নোটারাইজড সম্মতি ফর্ম এবং অন্যান্য প্রয়োজনীয় কাগজপত্র জমা দিতে হবে। ফি সহ মোট টাকা ১১,৪০৭ টাকা প্রদান করতে হবে এবং এক্সপ্রেস ডেলিভারি সুবিধা পাওয়া যাবে।